In [107]:
%env NX_CUGRAPH_AUTOCONFIG=True
import networkx as nx
from itertools import combinations
from collections import defaultdict, Counter
#import igraph as ig
import pandas as pd
import sqlite3
import random
from great_tables import GT

env: NX_CUGRAPH_AUTOCONFIG=True


In [3]:
conn = sqlite3.connect("tiktok_breadth_first.db")
cursor = conn.cursor()

In [4]:
query = """
SELECT hashtag_names
FROM videos
JOIN follow_relations
ON videos.reposter_username = follow_relations.from_username
WHERE follow_relations.to_username = 'kamalahq'
"""

kamalahq_docs = cursor.execute(query).fetchall()

In [16]:
query = """
SELECT hashtag_names
FROM videos
JOIN follow_relations
ON videos.reposter_username = follow_relations.from_username
WHERE follow_relations.to_username = 'teamtrump'
"""

teamtrump_docs = cursor.execute(query).fetchall()

In [5]:
kamalahq_token = Counter()
for doc in kamalahq_docs:
    contents = doc[0].lower().split(',')
    kamalahq_token.update(contents)

In [25]:
teamtrump_token = Counter()
for doc in teamtrump_docs:
    contents = doc[0].lower().split(',')
    teamtrump_token.update(contents)

In [27]:
kamalahq_edge_weights = defaultdict(int)
for doc in kamalahq_docs:
    contents = doc[0].lower().split(',')
    tokens = sorted(set(contents)) # sorted such that edge weight pairs are indexed alphabetically
    for u, v in combinations(tokens, 2):
        kamalahq_edge_weights[(u, v)] += 1

In [28]:
teamtrump_edge_weights = defaultdict(int)
for doc in teamtrump_docs:
    contents = doc[0].lower().split(',')
    tokens = sorted(set(contents)) # sorted such that edge weight pairs are indexed alphabetically
    for u, v in combinations(tokens, 2):
        teamtrump_edge_weights[(u, v)] += 1

In [ ]:
G_k = nx.Graph()
for (u, v), weight in kamalahq_edge_weights.items():
    G_k.add_edge(u, v, weight=weight)

In [29]:
G_t = nx.Graph()
for (u, v), weight in teamtrump_edge_weights.items():
    G_t.add_edge(u, v, weight=weight)

In [10]:
nx.write_weighted_edgelist(G_k, 'kamalahq_hashtag_network.edgelist')

In [ ]:
nx.write_weighted_edgelist(G_t, 'teamtrump_hashtag_network.edgelist')

In [4]:
G_k = nx.read_weighted_edgelist('kamalahq_hashtag_network.edgelist')

In [5]:
G_t = nx.read_weighted_edgelist('teamtrump_hashtag_network.edgelist')

In [14]:
import json

In [69]:
nx.config.backend_priority = ["cugraph", "networkx"]

In [70]:
k_degree = nx.degree_centrality(G_k)

In [7]:
k_file_path = "k_degree.json"
with open(k_file_path, 'w') as json_file:
    json.dump(k_degree, json_file, indent=4)

In [8]:
t_degree = nx.degree_centrality(G_t)

In [9]:
t_file_path = "t_degree.json"
with open(t_file_path, 'w') as json_file:
    json.dump(t_degree, json_file, indent=4)

In [6]:
random.seed(42)
k_betweenness = nx.betweenness_centrality(G_k, k=50)

In [21]:
k_file_path = "k_betweenness.json"
with open(k_file_path, 'w') as json_file:
    json.dump(k_betweenness, json_file, indent=4)

In [9]:
random.seed(42)
t_betweenness = nx.betweenness_centrality(G_t, k=50)

In [22]:
t_file_path = "t_betweenness.json"
with open(t_file_path, 'w') as json_file:
    json.dump(t_betweenness, json_file, indent=4)

In [82]:
k_pagerank = nx.pagerank(G_k, weight='weight')

In [83]:
k_file_path = "k_pagerank.json"
with open(k_file_path, 'w') as json_file:
    json.dump(k_pagerank, json_file, indent=4)

In [84]:
t_pagerank = nx.pagerank(G_t, weight='weight')

In [85]:
t_file_path = "t_pagerank.json"
with open(t_file_path, 'w') as json_file:
    json.dump(t_pagerank, json_file, indent=4)

In [86]:
measures = ['betweenness', 'degree', 'pagerank']
k_measures_df = pd.DataFrame({})
for m in measures:
    with open(f'k_{m}.json') as f:
        data = json.load(f)
    df = pd.DataFrame(data.values(), index=data.keys(), columns = [m])
    k_measures_df = pd.concat([k_measures_df, df], axis=1)

In [87]:
measures = ['betweenness', 'degree', 'pagerank']
t_measures_df = pd.DataFrame({})
for m in measures:
    with open(f't_{m}.json') as f:
        data = json.load(f)
    df = pd.DataFrame(data.values(), index=data.keys(), columns = [m])
    t_measures_df = pd.concat([t_measures_df, df], axis=1)

In [88]:
k_measures_df.to_csv('kamala_nodes_measure.csv')
t_measures_df.to_csv('trump_nodes_measure.csv')

In [3]:
from networkx import approximation

In [92]:
t_avg_cluster = approximation.average_clustering(G_t, trials=100000, seed=42)
k_avg_cluster = approximation.average_clustering(G_k, trials=100000, seed=42)

In [99]:
k_max_comp = max(nx.connected_components(G_k), key=len)
t_max_comp = max(nx.connected_components(G_t), key=len)

In [101]:
k_diameter = approximation.diameter(nx.induced_subgraph(G_k, k_max_comp), seed=42)
t_diameter = approximation.diameter(nx.induced_subgraph(G_t, t_max_comp), seed=42)

In [93]:
t_avg_cluster, k_avg_cluster

(0.84527, 0.84437)

In [94]:
t_density = nx.density(G_t)
k_density = nx.density(G_k)

In [43]:
t_density, k_density

(0.00014269010099396336, 8.91057180921317e-05)

In [44]:
t_nnodes = nx.number_of_nodes(G_t)
k_nnodes = nx.number_of_nodes(G_k)

In [45]:
t_nnodes, k_nnodes

(246620, 487200)

In [46]:
t_nedges = nx.number_of_edges(G_t)
k_nedges = nx.number_of_edges(G_k)

In [64]:
t_nedges, k_nedges

(4339290, 10575216)

In [126]:
stats_df = pd.DataFrame({
    'Metric': [
        'Number of Nodes',
        'Number of Edges',
        'Density',
        'Average Clustering Coeff',
        'Diameter (Maximal Component)'
        
    ],
    'Harris': [
        str(k_nnodes),
        str(k_nedges),
        f'{k_density:.4e}',
        k_avg_cluster,
        str(k_diameter)
    ],
    'Trump': [
        str(t_nnodes),
        str(t_nedges),
        f'{t_density:.4e}',
        t_avg_cluster,
        str(t_diameter)
    ]
})

In [125]:
stats_df.head()

,Metric,Harris,Trump
0,Number of Nodes,487200,246620
1,Number of Edges,10575216,4339290
2,Density,8.9106e-05,1.42690e-04
3,Average Clustering Coeff,0.84437,0.84527
4,Diameter (Maximal Component),11,9


In [127]:
basic_stats = GT(stats_df).tab_header(title='Networks Overview: Harris vs. Trump')
basic_stats.save('viz/gt_network_stats.png')

GT(_tbl_data=                         Metric      Harris       Trump
0               Number of Nodes      487200      246620
1               Number of Edges    10575216     4339290
2                       Density  8.9106e-05  1.4269e-04
3      Average Clustering Coeff     0.84437     0.84527
4  Diameter (Maximal Component)          11           9, _body=<great_tables._gt_data.Body object at 0x7f39d85ee050>, _boxhead=Boxhead([ColInfo(var='Metric', type=<ColInfoTypeEnum.default: 1>, column_label='Metric', column_align='left', column_width=None), ColInfo(var='Harris', type=<ColInfoTypeEnum.default: 1>, column_label='Harris', column_align='left', column_width=None), ColInfo(var='Trump', type=<ColInfoTypeEnum.default: 1>, column_label='Trump', column_align='left', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x7f39cabd5ad0>, _spanners=Spanners([]), _heading=Heading(title='Networks Overview: Harris vs. Trump', subtitle=None, preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x7f39cabc5610>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x7f39cabc4650>, _source_notes=[], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x7f39cabc5a10>, _formats=[], _substitutions=[], _col_merge=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_additional_css=OptionsInfo(scss=False, category='table', type='values', value=[]), table_font_names=OptionsInfo(scss=False, category='table', type='values', value=['-apple-system', 'BlinkMacSystemFont', 'Segoe UI', 'Roboto', 'Oxygen', 'Ubuntu', 'Cantarell', 'Helvetica Neue', 'Fira Sans', 'Droid Sans', 'Arial', 'sans-serif']), table_font_size=OptionsInfo(scss=True, category='table', type='px', value='16px'), table_font_weight=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_style=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_color=OptionsInfo(scss=True, category='table', type='value', value='#333333'), table_font_color_light=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_border_top_include=OptionsInfo(scss=False, category='table', type='boolean', value=True), table_border_top_style=OptionsInfo(scss=True, category='table', type='value', value='solid'), table_border_top_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_top_color=OptionsInfo(scss=True, category='table', type='value', value='#A8A8A8'), table_border_right_style=OptionsInfo(scss=True, category='table', type='value', value='none'), table_border_right_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_right_color=OptionsInfo(scss=True, category='table', type='value', value='#D3D3D3'), table_border_bottom_include=OptionsInfo(scss=False, category='table', type='boolean', value=True), table_border_bottom_style=OptionsInfo(scss=True, category='table', type='value', value='solid'), table_border_bottom_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_bottom_color=OptionsInfo(scss=True, category='table', type='value', value='#A8A8A8'), table_border_left_style=OptionsInfo(scss=True, category='table', type='value', value='none'), table_border_left_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_left_color=OptionsInfo(scss=True, category='table', type='value',

In [128]:
import seaborn as sns
import matplotlib.pyplot as plt

In [131]:
k_measures_df['candidate'] = ['kamalahq'] * len(k_measures_df)
t_measures_df['candidate'] = ['teamtrump'] * len(t_measures_df)

In [139]:
k_measures_df.reset_index(names='hashtag', inplace=True)
t_measures_df.reset_index(names='hashtag', inplace=True)

In [140]:
nodes_df = pd.concat([k_measures_df, t_measures_df])

In [141]:
nodes_df.head()

,hashtag,betweenness,degree,pagerank,candidate
0,ahs,1.887389e-05,0.001576,0.000025,kamalahq
1,blancaevangelista,1.689217e-07,0.000070,0.000001,kamalahq
2,edit,2.847784e-02,0.148003,0.006867,kamalahq
3,fypシ,3.650446e-02,0.159980,0.007019,kamalahq
4,posefx,3.073150e-06,0.000400,0.000009,kamalahq


In [146]:
nx.degree_histogram(G_k)

[0,
 7735,
 16601,
 25582,
 73094,
 18859,
 20332,
 19698,
 18529,
 16713,
 15564,
 14887,
 13629,
 12165,
 11400,
 10231,
 9708,
 8602,
 8292,
 8301,
 6819,
 6172,
 5764,
 5240,
 5193,
 4365,
 4484,
 4124,
 3795,
 4296,
 3295,
 3205,
 2956,
 2900,
 2472,
 2533,
 2387,
 2274,
 1980,
 2075,
 1960,
 1948,
 1607,
 1774,
 1543,
 1506,
 1409,
 1362,
 1392,
 1317,
 1276,
 1213,
 1239,
 1049,
 1051,
 1174,
 961,
 844,
 1002,
 885,
 760,
 894,
 830,
 885,
 714,
 680,
 691,
 712,
 753,
 706,
 739,
 674,
 685,
 592,
 637,
 673,
 564,
 569,
 592,
 539,
 506,
 453,
 435,
 514,
 613,
 688,
 527,
 345,
 432,
 416,
 409,
 511,
 403,
 367,
 365,
 345,
 330,
 353,
 341,
 420,
 388,
 335,
 296,
 300,
 364,
 257,
 274,
 387,
 285,
 297,
 278,
 246,
 282,
 290,
 258,
 225,
 286,
 213,
 264,
 242,
 234,
 249,
 235,
 199,
 192,
 272,
 204,
 210,
 164,
 183,
 185,
 186,
 155,
 180,
 148,
 210,
 164,
 163,
 133,
 173,
 147,
 140,
 138,
 168,
 150,
 231,
 196,
 211,
 130,
 136,
 128,
 153,
 136,
 128,
 146,
 1

In [154]:
fig = sns.barplot(
    x=range(len(nx.degree_histogram(G_k))),
    y=nx.degree_histogram(G_k)
)


KeyboardInterrupt: 

Error in callback <function _draw_all_if_interactive at 0x7f39c9b6dc60> (for post_execute), with arguments args (),kwargs {}:


KeyboardInterrupt: 

Error in callback <function flush_figures at 0x7f39c2cbc680> (for post_execute), with arguments args (),kwargs {}:


KeyboardInterrupt: 

In [ ]:
fig.show()

In [142]:
common_nodes = pd.merge(k_measures_df, t_measures_df, on='hashtag')

In [144]:
common_nodes.shape

(106941, 9)